---
title: "Part 11: Time Series"
---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sambaiga/ds-mlops-path/blob/main/tutorials/01-python-basics/11-time-series.ipynb) [![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue.svg?logo=jupyter&logoColor=white)](https://raw.githubusercontent.com/sambaiga/ds-mlops-path/main/tutorials/01-python-basics/11-time-series.ipynb)

**DS-MLOps Data Analysis**

**Python 3.12+ | Author: Anthony Faustine**

## Before you begin

This notebook continues from Part 10 (`10-combining-reshaping.ipynb`), which covered concatenation, merging, groupby, and pivoting. The student exam results dataset has no real dates in it, only an `establishment_year`, so this part introduces a second dataset built for the job: daily attendance records for 5 schools over a school term — the shape most time-indexed data takes in practice, one row per day per entity.

Part 12 (`11-polars.ipynb`) continues with Polars, including its own (faster) take on time-indexed data.

| Topic | Why it matters |
|---|---|
| **`Timestamp` and `to_datetime`** | pandas' building block for a single point in time, and how to parse one out of text |
| **`DatetimeIndex`** | An index made of dates unlocks date-based slicing, not just position-based |
| **Selecting by date** | Partial strings and date ranges, instead of exact label or integer position |
| **`resample`** | Change the time granularity of a series: daily to weekly, weekly to monthly |

> Callout markers used throughout this notebook are explained on the [book cover page](../../index.qmd#callout-guide).

## Learning Objectives

By the end of Part 4 you will be able to:

| # | Skill | Covered in |
|---|---|---|
| 1 | Create and inspect `Timestamp` values, and parse dates out of text with `to_datetime` | Sec. 1 |
| 2 | Build a `DatetimeIndex` and use it to slice a time series by date | Sec. 2 |
| 3 | Select rows with a partial date string or a date range | Sec. 3 |
| 4 | Change the time granularity of a series with `resample` | Sec. 4 |

In [ ]:
import numpy as np
import pandas as pd

attendance = pd.read_csv("data/daily_attendance.csv")
attendance.dtypes

## 1. Date and Time Data Types

The `date` column above read in as plain text, `str` dtype, not a date pandas can do arithmetic on. `pd.to_datetime` converts it to pandas' dedicated datetime dtype, `datetime64`:

In [ ]:
attendance["date"] = pd.to_datetime(attendance["date"])
attendance.dtypes

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: Pandas 3 infers the resolution it needs, not always nanoseconds</span><br><br>
Earlier pandas versions always stored datetimes as <code>datetime64[ns]</code>, nanosecond precision, whether the data needed it or not. Pandas 3's <code>to_datetime</code> infers a resolution from what is actually in the data: day-level strings like the ones here become <code>datetime64[s]</code> or coarser, not nanoseconds. <code>attendance["date"].dtype</code> shows whichever resolution was inferred for this column.
</div>

A single value out of a datetime column is a `Timestamp`, pandas' equivalent of Python's `datetime.datetime`, with the same year, month, day, and weekday attributes:

In [ ]:
first_day = attendance["date"].iloc[0]
print(type(first_day))
print(f"year={first_day.year}, month={first_day.month}, day_name={first_day.day_name()}")

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 1 - Parse and Inspect</span><br><br>

<b>Goal:</b> Convert a list of three date strings, <code>["2025-01-06", "2025-02-14", "2025-03-28"]</code>, to <code>Timestamp</code> values with <code>pd.to_datetime</code>, then print the day name of each one.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>dates = pd.to_datetime(["2025-01-06", "2025-02-14", "2025-03-28"])
for d in dates:
    print(d.day_name())</pre>
</div>

In [ ]:
# TODO: convert the three date strings and print each day name
...

## 2. The `DatetimeIndex`

Setting a datetime column as the index turns it into a `DatetimeIndex`, which unlocks date-based slicing instead of only position-based or exact-label lookups. Each school's rows are pulled out first, since a `DatetimeIndex` only makes sense for one time series at a time:

In [ ]:
school_300 = attendance[attendance["school_id"] == 300].set_index("date")
school_300.index

<div style='background:#FEF2F2;border-left:5px solid #DC2626;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#991B1B;font-weight:bold'><i class="bi bi-bug-fill"></i> Common Mistake: Setting the index before filtering to one entity</span><br><br>
<code>attendance.set_index("date")</code> on the full table produces a <code>DatetimeIndex</code> with the same date repeated once per school, since every school has a row for every day. Slicing that index by date then returns rows from every school mixed together for that date, not a clean single time series. Filter to one entity first, exactly as <code>school_300</code> does above, then set the index.
</div>

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 2 - Build Another School's Series</span><br><br>

<b>Goal:</b> Filter <code>attendance</code> to <code>school_id == 302</code>, set <code>date</code> as the index, and confirm the result's index is a <code>DatetimeIndex</code> with <code>isinstance(result.index, pd.DatetimeIndex)</code>.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>school_302 = attendance[attendance["school_id"] == 302].set_index("date")
isinstance(school_302.index, pd.DatetimeIndex)</pre>
</div>

In [ ]:
# TODO: filter to school_id 302, set date as index, confirm DatetimeIndex
...

## 3. Selecting Data from a Time Series

A `DatetimeIndex` accepts a partial date string in `.loc`, matching every row that falls inside it. `"2025-02"` selects the whole month without spelling out the first and last day:

In [ ]:
school_300.loc["2025-02"].head()

A slice with two partial dates selects everything between them, inclusive of both ends:

In [ ]:
school_300.loc["2025-02-01":"2025-02-07"]

<div style='background:#EAF7F0;border-left:5px solid #059669;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#059669;font-weight:bold'><i class="bi bi-journal-code"></i> Example: Comparing the size of two date ranges</span><br><br>
<code>len(school_300.loc["2025-01"])</code> against <code>len(school_300.loc["2025-02"])</code> confirms the row count for each month matches its number of business days, the same `bdate_range` weekday-only pattern used to build this dataset in the first place.
</div>

In [ ]:
print(f"January business days  : {len(school_300.loc['2025-01'])}")
print(f"February business days : {len(school_300.loc['2025-02'])}")

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 3 - Filter the Last Two Weeks of Term</span><br><br>

<b>Goal:</b> Select every row in <code>school_300</code> from <code>"2025-03-15"</code> to <code>"2025-03-28"</code> inclusive, and print the mean <code>attendance_rate</code> over that range.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>last_two_weeks = school_300.loc["2025-03-15":"2025-03-28"]
last_two_weeks["attendance_rate"].mean()</pre>
</div>

In [ ]:
# TODO: select 2025-03-15 through 2025-03-28 and print the mean attendance_rate
...

## 4. The Power of Pandas: `resample`

`resample` changes the time granularity of a series: daily data summarized into weekly or monthly figures, the same split-apply-combine idea from Part 3's `groupby`, except the groups are time intervals instead of category values:

In [ ]:
weekly_attendance = school_300["attendance_rate"].resample("W").mean()
weekly_attendance.head()

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: <code>resample</code> groups by time interval, <code>groupby</code> groups by value</span><br><br>
<code>df.groupby("caste")</code> (Part 3) splits rows by whatever value is already in the <code>caste</code> column. <code>series.resample("W")</code> splits rows by which week their <code>DatetimeIndex</code> label falls into, intervals that did not exist as a column at all until <code>resample</code> created them. Both still end with an aggregation like <code>.mean()</code> to combine each group into one number.
</div>

Monthly resampling on the same series needs only a different frequency string. In pandas 3 the old `"M"` alias was removed; the replacement is `"ME"` (month-end), which anchors each bucket to the last calendar day of the month:

In [ ]:
monthly_attendance = school_300["attendance_rate"].resample("ME").mean()
monthly_attendance

<div style='background:#F5F3FF;border-left:5px solid #7C3AED;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#5B21B6;font-weight:bold'><i class="bi bi-lightbulb-fill"></i> Pro Tip: Resampling the whole DataFrame keeps every entity separate, with care</span><br><br>
<code>attendance.set_index("date").groupby("school_id")["attendance_rate"].resample("W").mean()</code> resamples each school's series independently in one call, instead of looping over schools and resampling each one by hand. <code>groupby</code> before <code>resample</code> is what keeps the schools from being averaged together.
</div>

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 4 - Monthly Comparison Across Schools</span><br><br>

<b>Goal:</b> Set <code>date</code> as the index on the full <code>attendance</code> table, group by <code>school_id</code>, and resample to monthly means in one chained call. Use <code>"ME"</code> (month-end) — the pandas 3 replacement for the removed <code>"M"</code> alias.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>attendance.set_index("date").groupby("school_id")["attendance_rate"].resample("ME").mean()</pre>
</div>

In [ ]:
# TODO: set date as index, group by school_id, resample monthly, take the mean
...

## Capstone: Term-End Attendance Report

Combine every operation from this notebook: parsing dates, building a per-school time series, slicing by date, and resampling, into one short report comparing the start and end of term.

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Capstone Exercise - Term-End Attendance Report</span><br><br>

<b>Goal:</b>
<ol>
<li>Set <code>date</code> as the index on the full <code>attendance</code> table (Sec. 2)</li>
<li>Group by <code>school_id</code> and resample to weekly means (Sec. 4)</li>
<li>From the result, select the first week of January and the last week of March for every school, using a partial date string (Sec. 3)</li>
<li>Report which school had the largest drop in attendance between those two weeks</li>
</ol>
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>weekly = attendance.set_index("date").groupby("school_id")["attendance_rate"].resample("W").mean()

first_week = weekly.loc[:, "2025-01-06":"2025-01-12"]
last_week = weekly.loc[:, "2025-03-24":"2025-03-28"]
drop_per_school = first_week.groupby("school_id").mean() - last_week.groupby("school_id").mean()
drop_per_school.sort_values(ascending=False)</pre>
</div>

In [ ]:
# TODO: build the term-end attendance report described above
...

## Further Reading

| Resource | Why it matters |
|---|---|
| McKinney, W. (2022). *Python for Data Analysis*, 3rd ed. O'Reilly. | Chapter 11 (time series) is the most complete treatment of `DatetimeIndex`, `resample`, and `rolling` with pandas |
| [pandas documentation — Time series / date functionality](https://pandas.pydata.org/docs/user_guide/timeseries.html) | The authoritative reference for `pd.date_range`, offset aliases, time zones, and `Period` |
| McKinney, W. (2011). [Time series analysis in Python with pandas](https://conference.scipy.org/proceedings/scipy2011/pdfs/wes_mckinney.pdf). *SciPy 2011*. | The original paper describing pandas' time series design; short and still accurate |
| Hyndman, R.J. & Athanasopoulos, G. (2021). *Forecasting: Principles and Practice*, 3rd ed. | Free at [otexts.com/fpp3](https://otexts.com/fpp3) — the next step after understanding how to *store* time series data is learning how to *model* it |


## Summary

| Concept | Key rule |
|---|---|
| `pd.to_datetime` | Parses text into pandas' `datetime64` dtype; pandas 3 infers the resolution instead of always using nanoseconds |
| `Timestamp` | A single datetime value, with `.year`, `.month`, `.day_name()`, and similar attributes |
| `DatetimeIndex` | Set a datetime column as the index to unlock date-based slicing |
| Filter before indexing | Set a `DatetimeIndex` on one entity's rows, not a table mixing several entities at the same dates |
| `.loc["2025-02"]` | A partial date string selects every row inside that period |
| `.loc[start:end]` | A date range slice is inclusive of both ends |
| `.resample("ME")` | Groups rows by month-end interval; `"ME"` is the pandas 3 replacement for the removed `"M"` alias |
| `groupby(...).resample(...)` | Resample each entity's series independently in one chained call |

**Next:** `11-polars.ipynb`, covering Polars' DataFrame, expression API, and lazy evaluation.